In [1]:
import json


input_file = '/home/XXX/CodeSemantic/CodeSemantic/statement_Accuracy_Results/statement_results.jsonl'
output_file = 'filtered_results.jsonl'

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        data = json.loads(line)
        if (data['Incontext'] == 'different' and 
            data['CoT'] == 'no' and 
            data['shot'] == 3 and 
            data['quantization'] == 'no'):
            json.dump(data, outfile)
            outfile.write('\n')  # Add newline for JSONL format

In [8]:
import json
import pandas as pd
from collections import defaultdict

def parse_jsonl(file_path):
    """Robust JSONL parser that handles malformed lines"""
    data = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:  # Skip empty lines
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line {i+1}: {e}")
                print(f"Problematic line content: {line[:100]}...")
    return data

# Initialize data structures
assignment_data = defaultdict(dict)
branch_data = defaultdict(dict)

# Parse the file
entries = parse_jsonl('filtered_results.jsonl')

if not entries:
    print("No valid JSON entries found in the file!")
else:
    for entry in entries:
        model = entry.get('Model')
        prompt = entry.get('Prompt')
        
        if not model or not prompt:
            print(f"Skipping entry missing Model/Prompt: {entry}")
            continue
            
        type_acc = entry.get('type_accuracy', {})
        
        # Collect Assignment accuracies
        if 'Assignment' in type_acc:
            assignment_data[model][f'PT{prompt} Assignment Accuracy'] = type_acc['Assignment']
        
        # Collect Branch accuracies (Boolean)
        if 'Branch' in type_acc:
            branch_data[model][f'PT{prompt} Boolean Accuracy'] = type_acc['Branch']

    # Create DataFrames with specified column order
    def create_df(data_dict, col_order):
        df = pd.DataFrame.from_dict(data_dict, orient='index')
        # Ensure all requested columns exist (add as NaN if missing)
        for col in col_order:
            if col not in df.columns:
                df[col] = None
        return df[col_order].reset_index().rename(columns={'index': 'Model Name'})

    # Define column orders
    assignment_cols = [
        'PT1 Assignment Accuracy', 
        'PT5 Assignment Accuracy', 
        'PT6 Assignment Accuracy'
    ]
    
    branch_cols = [
        'PT1 Boolean Accuracy',
        'PT5 Boolean Accuracy', 
        'PT6 Boolean Accuracy',
        'PT7 Boolean Accuracy'
    ]
    
    # Create DataFrames
    df_assignment = create_df(assignment_data, assignment_cols)
    df_branch = create_df(branch_data, branch_cols)

    # Save to Excel using pandas' default writer
    try:
        with pd.ExcelWriter('model_accuracies.xlsx') as writer:
            df_assignment.to_excel(writer, sheet_name='Assignment Accuracies', index=False)
            df_branch.to_excel(writer, sheet_name='Boolean Accuracies', index=False)
        print("Successfully created model_accuracies.xlsx with PT7 Boolean Accuracy included")
    except Exception as e:
        print(f"Error saving Excel file: {e}")

Successfully created model_accuracies.xlsx with PT7 Boolean Accuracy included


# Abstract Value Random Baseline

In [1]:
import json


input_file = '/home/XXX/CodeSemantic/CodeSemantic/statement_Accuracy_Results/statement_results.jsonl'
output_file = 'filtered_results_quantized.jsonl'

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        data = json.loads(line)
        if (data['Incontext'] == 'different' and 
            data['CoT'] == 'no' and 
            data['quantization'] == 'yes'):
            json.dump(data, outfile)
            outfile.write('\n')  # Add newline for JSONL format

In [ ]:
import pandas as pd
import json

def parse_jsonl(file_path):
    """Robustly parses a JSONL file and skips bad lines."""
    data = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line {i+1}: {e}")
                print(f"Problematic line: {line[:80]}...")
    return data

# Load your JSONL file
input_file = 'filtered_results_quantized.jsonl'  
entries = parse_jsonl(input_file)

# Collect model-shot-accuracy tuples
records = []
for entry in entries:
    model = entry.get("Model")
    shot = entry.get("shot")
    accuracy = entry.get("accuracy")
    if model is not None and shot is not None and accuracy is not None:
        records.append((model, shot, accuracy))

# Build DataFrame
df = pd.DataFrame(records, columns=["Model", "Shot", "Accuracy"])

# Pivot table: rows = Model, columns = Shot values
pivot = df.pivot_table(index="Model", columns="Shot", values="Accuracy")

# Rename columns to readable format
pivot.columns = [f"{int(c)}-shot" for c in pivot.columns]

# Ensure all 0–3 shot columns exist
for col in ["0-shot", "1-shot", "2-shot", "3-shot"]:
    if col not in pivot.columns:
        pivot[col] = None

# Add Model Name column and reorder
pivot["Model Name"] = pivot.index
final_df = pivot[["Model Name", "0-shot", "1-shot", "2-shot", "3-shot"]]

# Save to Excel
final_df.to_excel("model_shot_accuracies.xlsx", index=False)
print("✅ Excel file saved as model_shot_accuracies.xlsx")


✅ Excel file saved as model_shot_accuracies.xlsx


In [5]:
import pandas as pd
import json

def parse_jsonl(file_path):
    """Robustly parses a JSONL file and skips bad lines."""
    data = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error parsing line {i+1}: {e}")
                print(f"Problematic line: {line[:80]}...")
    return data

# Load your JSONL file
input_file = '/home/XXX/CodeSemantic/CodeSemantic/statement_Accuracy_Results/statement_python_rebuttal_results_yes.jsonl'  
entries = parse_jsonl(input_file)

# Collect model-shot-accuracy tuples
records = []
for entry in entries:
    model = entry.get("Model")
    shot = entry.get("shot")
    accuracy = entry.get("accuracy")
    if model is not None and shot is not None and accuracy is not None:
        records.append((model, shot, accuracy))

# Build DataFrame
df = pd.DataFrame(records, columns=["Model", "Shot", "Accuracy"])

# Pivot table: rows = Model, columns = Shot values
pivot = df.pivot_table(index="Model", columns="Shot", values="Accuracy")

# Rename columns to readable format
pivot.columns = [f"{int(c)}-shot" for c in pivot.columns]

# Ensure all 0–3 shot columns exist
for col in ["0-shot", "1-shot", "2-shot", "3-shot"]:
    if col not in pivot.columns:
        pivot[col] = None

# Add Model Name column and reorder
pivot["Model Name"] = pivot.index
final_df = pivot[["Model Name", "0-shot", "1-shot", "2-shot", "3-shot"]]

# Save to Excel
final_df.to_excel("model_shot_accuracies_random.xlsx", index=False)
print("✅ Excel file saved as model_shot_accuracies.xlsx")


✅ Excel file saved as model_shot_accuracies.xlsx


In [13]:
import json

def has_dict_with_size_gt_2(obj):
    """Check if any dict (including nested) has > 2 keys."""
    if isinstance(obj, dict):
        if len(obj) >= 2:
            return True
        # Recursively check nested dicts
        return any(has_dict_with_size_gt_2(v) for v in obj.values())
    elif isinstance(obj, list):
        # Check all elements in lists
        return any(has_dict_with_size_gt_2(v) for v in obj)
    return False

count = 0

with open("/home/XXX/CodeSemantic/CodeSemantic/dataset/block_analysis_c.jsonl", "r") as file:
    for line in file:
        entry = json.loads(line)
        value_after = entry.get("Value After Statement Execution")
        #print(value_after)
        
        if isinstance(value_after, dict) and has_dict_with_size_gt_2(value_after):
            print(entry.get('Block_Size'))
            count += 1

print(f"Total entries with a dictionary (>2 keys at any level): {count}")

32
12
34
68
69
74
12
9
13
20
25
30
23
3
10
Total entries with a dictionary (>2 keys at any level): 15


In [15]:
import json

def has_dict_with_size_gt_2(obj):
    """Check if any dict (including nested) has > 2 keys."""
    if isinstance(obj, dict):
        if len(obj) >= 2:
            return True
        # Recursively check nested dicts
        return any(has_dict_with_size_gt_2(v) for v in obj.values())
    elif isinstance(obj, list):
        # Check all elements in lists
        return any(has_dict_with_size_gt_2(v) for v in obj)
    return False

count = 0

with open("/home/XXX/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_C.jsonl", "r") as file:
    for line in file:
        data = json.loads(line)
        
        idx = data.get('idx', 'N/A')
        source_code = data.get('Source Code', 'N/A')
        statement_type = data.get('Statement Type', 'N/A')
        selected_statement = data.get('Selected Statement', 'N/A')
        function_input = data.get('Function Input', 'N/A')
        value_before_execution = data.get('Variable Values Before Statement', 'N/A')
        value_after_execution = data.get('Value After Statement Execution', 'N/A')
        value_after = data.get("Value After Statement Execution")

        # if statement_type == 'Constant Assignment' or statement_type == 'Branch':
        #     continue 
        # print(f'IDX:{idx}')
        # print("Source Code:")
        # print(source_code)
        # print("\nSelected Statement:")
        # print(selected_statement)
        # print("\nFunction Input:")
        # print(function_input)
        # print("\nValue before Execution:")
        # print(value_before_execution)
        # print("\nValue After Execution:")
        # print(value_after_execution)
        # print("\n" + "="*50 + "\n")  # Separator between entries
        
        if isinstance(value_after, dict) and has_dict_with_size_gt_2(value_after):
            count += 1

print(f"Total entries with a dictionary (>2 keys at any level): {count}")

Total entries with a dictionary (>2 keys at any level): 7


In [20]:
import json

def has_dict_with_size_gt_2(obj):
    """Check if any dict (including nested) has > 2 keys."""
    if isinstance(obj, dict):
        if len(obj) >= 2:
            return True
        # Recursively check nested dicts
        return any(has_dict_with_size_gt_2(v) for v in obj.values())
    elif isinstance(obj, list):
        # Check all elements in lists
        return any(has_dict_with_size_gt_2(v) for v in obj)
    return False

count_has_dict_gt_2 = 0
missing_value_after = []
all_entries_count = 0

with open("/home/XXX/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_C_new.jsonl", "r") as file:
    for line in file:
        data = json.loads(line)
        all_entries_count += 1
        
        idx = data.get('idx', 'N/A')
        source_code = data.get('Source Code', 'N/A')
        statement_type = data.get('Statement Type', 'N/A')
        selected_statement = data.get('Selected Statement', 'N/A')
        function_input = data.get('Function Input', 'N/A')
        value_before_execution = data.get('Variable Values Before Statement', 'N/A')
        value_after_execution = data.get('Value After Statement Execution', 'N/A')
        value_after = data.get("Value After Statement Execution")

        # Check if "Value After Statement Execution" is missing or None
        if value_after is None or value_after == 'N/A':
            missing_value_after.append(idx)
        
        if isinstance(value_after, dict) and has_dict_with_size_gt_2(value_after):
            count_has_dict_gt_2 += 1

print(f"Total entries: {all_entries_count}")
print(f"Total entries with a dictionary (>2 keys at any level): {count_has_dict_gt_2}")
print(f"Total entries missing 'Value After Statement Execution': {len(missing_value_after)}")

if missing_value_after:
    print("\nEntries missing 'Value After Statement Execution':")
    for idx in missing_value_after:
        print(f"  - IDX: {idx}")
else:
    print("\nAll entries have 'Value After Statement Execution'")

Total entries: 485
Total entries with a dictionary (>2 keys at any level): 0
Total entries missing 'Value After Statement Execution': 0

All entries have 'Value After Statement Execution'


In [16]:
import json

def has_dict_with_size_gt_2(obj):
    """Check if any dict (including nested) has > 2 keys."""
    if isinstance(obj, dict):
        if len(obj) >= 2:
            return True
        # Recursively check nested dicts
        return any(has_dict_with_size_gt_2(v) for v in obj.values())
    elif isinstance(obj, list):
        # Check all elements in lists
        return any(has_dict_with_size_gt_2(v) for v in obj)
    return False

input_path = "/home/XXX/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_C.jsonl"
output_path = "/home/XXX/CodeSemantic/CodeSemantic/dataset/statement_prediction_dataset_C_new.jsonl"

removed_count = 0
kept_count = 0

with open(input_path, "r") as infile, open(output_path, "w") as outfile:
    for line in infile:
        data = json.loads(line)
        value_after = data.get("Value After Statement Execution")

        if isinstance(value_after, dict) and has_dict_with_size_gt_2(value_after):
            removed_count += 1
            continue  # Skip writing this entry

        outfile.write(json.dumps(data) + "\n")
        kept_count += 1

print(f"Finished filtering.")
print(f"Entries removed (value_after dict > 7 keys): {removed_count}")
print(f"Entries kept: {kept_count}")


Finished filtering.
Entries removed (value_after dict > 7 keys): 7
Entries kept: 485
